Bibliotecas

In [5]:
# Importar las bibliotecas necesarias
from langchain_openai import ChatOpenAI
from openai import OpenAI
import os

# Verificar que tenemos las bibliotecas correctas
print("OpenAI library version:", __import__('openai').__version__)
print("Python version:", __import__('sys').version)

# Configuración del cliente OpenAI para GitHub Models
try:
    # Configurar el cliente con variables de entorno
    client = OpenAI(
        base_url=os.environ.get("GITHUB_BASE_URL"),
        api_key=os.environ.get("GITHUB_TOKEN")
    )
    
    # Verificar configuración (sin mostrar la API key completa por seguridad)
    print("Base URL configurada:", client.base_url)
    print("API Key configurada:", "✓" if client.api_key else "✗")
    
    if client.api_key:
        print("API Key preview:", client.api_key[:10] + "..." + client.api_key[-4:])
    else:
        print("⚠️  API Key no encontrada. Asegúrate de configurar GITHUB_TOKEN")
        
except Exception as e:
    print(f"Error en configuración: {e}")
    print("Verifica que las variables de entorno estén configuradas correctamente")


# Configuración del modelo LangChain con GitHub Models
try:
    llm = ChatOpenAI(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o",
        temperature=0.4,
        max_tokens=800
    )
    
    print("✓ Modelo LangChain configurado correctamente")
    print(f"Modelo: {llm.model_name}")
    print(f"Temperature: {llm.temperature}")
    print(f"Max tokens: {llm.max_tokens}")
    
except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Ver  ifica las variables de entorno OPENAI_BASE_URL y GITHUB_TOKEN")

OpenAI library version: 2.29.0
Python version: 3.12.5 (tags/v3.12.5:ff3bc82, Aug  6 2024, 20:45:27) [MSC v.1940 64 bit (AMD64)]
Base URL configurada: https://models.inference.ai.azure.com
API Key configurada: ✓
API Key preview: github_pat...PLIR
✓ Modelo LangChain configurado correctamente
Modelo: gpt-4o
Temperature: 0.4
Max tokens: 800


HardiBot primera version

In [1]:
import os
import time
from typing import Dict, List
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage

from rich.console import Console
from rich.markdown import Markdown
from rich.rule import Rule
from rich.live import Live

# Carga de variables de entorno 
load_dotenv()

# =====================================================================
# 1. CONFIGURACIÓN DEL MODELO 
# =====================================================================
_GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://models.inference.ai.azure.com")

if not _GITHUB_TOKEN:
    raise ValueError("CRÍTICO: GITHUB_TOKEN no encontrado. Violación de política Zero Trust.")

llm = ChatOpenAI(
    base_url=_BASE_URL,
    api_key=_GITHUB_TOKEN,
    model="gpt-4o",        
    temperature=0.4,         
    max_tokens=1200,
    streaming=True          
)

console = Console()
console.print("[bold green]✓ Modelo LangChain configurado y securizado correctamente.[/bold green]")

# =====================================================================
# 2. GESTIÓN DE MEMORIA Y SESIONES 
# =====================================================================
class WindowChatMessageHistory(BaseChatMessageHistory):
    """
    Patrón: Ventana Deslizante (Sliding Window).
    Mantiene solo los últimos 'k' intercambios para evitar latencia y exceso de tokens.
    """
    def __init__(self, k: int = 4) -> None:
        self.k: int = k
        self._messages: List[BaseMessage] = []
    
    @property
    def messages(self) -> List[BaseMessage]:
        return self._messages[-(self.k * 2):]
    
    def add_message(self, message: BaseMessage) -> None:
        self._messages.append(message)
    
    def clear(self) -> None:
        self._messages.clear()

# Almacén de sesiones en memoria
session_store: Dict[str, BaseChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Fábrica de historiales por ID de sesión."""
    if session_id not in session_store:
        session_store[session_id] = WindowChatMessageHistory(k=4)
    return session_store[session_id]

# =====================================================================
# 3. CONSTRUCCIÓN DE CADENA Y CONTEXTO DE PROMPT
# =====================================================================
system_prompt: str = """Eres HardiBot, un consultor inteligente de hardware especializado en el mercado chileno.
Tu objetivo es asesorar en el armado de PCs.
Reglas estrictas:
1. Las recomendaciones deben ajustarse al presupuesto en CLP (Pesos Chilenos).
2. Usa referencias locales (SPDigital, PC Factory, Winpy).
3. Evalúa compatibilidad técnica (ej. Sockets AM4 vs AM5, consumo energético).
4. Mantén un tono técnico, amigable y profesional."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Composición de la cadena acoplada a la memoria
conversation_chain = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# =====================================================================
# 4. CAPA DE PRESENTACIÓN Y STREAMING CON RICH
# =====================================================================
def chat_hardibot(user_input: str, session_id: str = "hardibot_eval_1") -> None:
    """Maneja el flujo I/O aplicando Streaming Chunk-by-Chunk."""
    console.print(Rule(title="🤖 HardiBot", style="bold blue", align="left"))
    
    respuesta_acumulada: str = ""
    chunk_count: int = 0
    start_time: float = time.time()
    
    try:
        # Uso de Rich Live para actualizar la UI sin bloqueos visuales
        with Live(Markdown("⏳ *Analizando requerimientos...*"), console=console, refresh_per_second=15) as live:
            for chunk in conversation_chain.stream(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            ):
                token = chunk.content
                respuesta_acumulada += token
                chunk_count += 1
                live.update(Markdown(respuesta_acumulada))
                
    except Exception as e:
        console.print(f"\n[bold red]❌ Error de Inferencia:[/bold red] {e}")
        return

    # Telemetría de desempeño
    tiempo_total = time.time() - start_time
    console.print(Rule(style="dim"))
    console.print(f"[dim]⚡ Procesado en {tiempo_total:.2f}s | {chunk_count} tokens generados.[/dim]")

def iniciar_hardibot() -> None:
    print("=" * 60)
    print(" 🛠️  HardiBot CLI - Modo Evaluación (IL1.1 Consolidado)")
    print(" Escribe 'salir' para finalizar")
    print("=" * 60)

    while True:
        try:
            user_input = input("\n👤 Tú: ").strip()
            if not user_input:
                continue
            if user_input.lower() in ["salir", "exit", "quit"]:
                print("\n👋 Sistema fuera de línea. Hardware asegurado.")
                break
            
            chat_hardibot(user_input)
            
        except KeyboardInterrupt:
            print("\n\n👋 Cierre forzado detectado. Hasta luego.")
            break

if __name__ == "__main__":
    iniciar_hardibot()

✓ Modelo LangChain configurado y securizado correctamente.

 🛠️  HardiBot CLI - Modo Evaluación (IL1.1 Consolidado)
 Escribe 'salir' para finalizar

👋 Sistema fuera de línea. Hardware asegurado.
